# 2 · Tree of Thought — Try Three, Keep The Best

**The problem:** ask a model for a recommendation and it commits to the **first idea**
that comes out. It never compared that idea with anything — by the third word,
it was already committed.

**The fix:** make it produce three *different* options, score each one **separately**,
and let your own code pick the highest score. That is Tree of Thought.

**The scenario for this whole notebook:**

> You are organising the annual company offsite.
> 40 people, one full day, budget ₹4,000 per person.
> You must recommend a venue type to management — **with reasons**.

Two examples:

1. The lazy way — one message (and why it fools you)
2. The proper way — generate, score, pick

In [ ]:
# ---- Step 0: check the kernel, then install what is missing ----
# Run this first. It works on Colab, on a fresh laptop,
# and it tells you plainly if the notebook is running the wrong Python.

import importlib.util, subprocess, sys

if sys.version_info < (3, 10):
    print("STOP - this notebook needs Python 3.10 or newer.")
    print("This kernel is Python", sys.version.split()[0], "at", sys.executable)
    print()
    print("Fix it like this:")
    print("  In Jupyter / VS Code : Kernel > Change Kernel, and pick the one from")
    print("                         structured_prompting/.venv")
    print("  On Colab             : Runtime > Restart session, then run this cell again")
    raise SystemExit("Wrong Python version - see the message above.")

REQUIRED = [
    ("openai", "openai==2.53.0"),
    ("dotenv", "python-dotenv==1.2.2"),
]

missing = [pkg for mod, pkg in REQUIRED if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    print("(a minute the first time, nothing the next time)")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done.")
else:
    print("All libraries already here.")

print("Python", sys.version.split()[0], "at", sys.executable)


In [ ]:
# ---- Setup: run this cell first ----
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    try:
        # Google Colab: add OPENAI_API_KEY in the Secrets panel (the key icon, left)
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        import getpass  # last resort: type it here, it is not saved anywhere
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI key: ")

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def ask(prompt, temperature=0):
    """Send one prompt to the model and return its reply as plain text."""
    reply = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return reply.choices[0].message.content

print("Setup done. Using model:", MODEL)

Setup done. Using model: gpt-4o-mini


---
## Example 1 · The lazy way — one message

**We want:** three options compared fairly.

**What we actually get:** watch closely.

In [ ]:
lazy_prompt = """We are organising an annual company offsite.
40 people, one full day, budget Rs 4,000 per person.

Give me 3 venue options and tell me which one is best."""

print(ask(lazy_prompt))

Look at how the reply is written. Almost always:

- Option 1 gets a warm, detailed description — and gets recommended.
- Options 2 and 3 are short and half-hearted. They were **written to lose**.

The model decided its answer while writing option 1, then produced two weak
options to make that answer look considered. This is not a comparison —
it is a decision wearing a comparison's clothes.

> **Lesson:** one message cannot both invent options and judge them fairly.

---
## Example 2 · The proper way — generate, score, pick

Three separate steps:

1. **Generate** — one call that produces 3 different options, with no favourite allowed.
2. **Score** — a fresh call *per option*. The scorer never sees the other options.
3. **Pick** — our own Python code takes the highest score.

### Step 1 — generate three different options

In [ ]:
generate_prompt = """We are organising an annual company offsite.
40 people, one full day, budget Rs 4,000 per person.

Give me 3 GENUINELY DIFFERENT venue options — a different type of venue,
not the same idea worded three ways.

For each option: a name for the option and 2 lines describing it.
Argue for each one equally strongly.
Do NOT say which one you prefer.

Separate the three options with a line containing only ###"""

reply = ask(generate_prompt, temperature=0.7)   # a little creativity helps variety
options = [part.strip() for part in reply.split("###") if part.strip()]

for i, option in enumerate(options, start=1):
    print(f"--- Option {i} ---")
    print(option)
    print()

### Step 2 — score each option in a separate, fresh call

Two important details:

- Each option is scored **alone**. The scorer does not know the others exist,
  so it cannot play favourites.
- The rubric says **what 0 and 1 mean**. Without that, every option gets a polite
  0.7 and the scores tell you nothing.

In [ ]:
def score_option(option_text):
    scoring_prompt = f"""You are scoring ONE venue option for a company offsite
(40 people, one day, Rs 4,000 per person). You are not comparing it to anything.

The option:
{option_text}

Score it on each point, from 0 to 1:
- budget fit:      1 = comfortably inside Rs 4,000 per person, 0 = clearly over
- travel:          1 = under 1 hour for most staff, 0 = half the day spent travelling
- facilities:      1 = space for both meetings and team activities, 0 = neither

Give one line of reasoning per point.
Then the average on the last line, exactly like this:
SCORE: 0.7"""

    reply = ask(scoring_prompt)
    last_line = reply.strip().splitlines()[-1]
    score = float(last_line.replace("SCORE:", "").strip())
    return score, reply

scores = []
for i, option in enumerate(options, start=1):
    score, full_reply = score_option(option)
    scores.append(score)
    print(f"--- Option {i} scored {score} ---")
    print(full_reply)
    print()

### Step 3 — our code picks the winner (not the model)

In [ ]:
best = scores.index(max(scores))

print("WINNER — option", best + 1, "with score", scores[best])
print()
print(options[best])
print()
print("Rejected, and why that matters for your email to management:")
for i, (option, score) in enumerate(zip(options, scores)):
    if i != best:
        first_line = option.splitlines()[0]
        print(f"  - Option {i+1} ({first_line}) scored {score}")

The recommendation email now writes itself: *"We recommend X. We also considered
Y and Z — here are their scores and why they lost."* The rejected options and
their reasons are often the most valuable part of the output.

---
## Summary

1. **What it is:** make 3 different options, score each one separately, pick the best in code.
2. **Never** ask for the options and the winner in the same message.
3. **Anchor the scores** — say what 0 and 1 mean, or everything scores 0.7.
4. **Keep the losers.** "We considered and rejected..." is what makes a
   recommendation convincing.

**Try it yourself:** change the scenario to choosing a training provider or
a new office location. Only the two prompts need to change.